In [1]:
# Useful for debugging
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import scipy.constants as spc
import subprocess
import os, re
from skopt import gp_minimize
from skopt.space import Real
from skopt.plots import plot_convergence, plot_objective, plot_evaluations, plot_gaussian_process
from functools import partial
import pandas as pd
import pickle
import multiprocessing as mp

In [3]:
# Nicer plotting
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12,8)
%config InlineBackend.figure_format = 'retina'

In [4]:
from astra import Astra
# load astra and generator binaries
%env ASTRA_BIN=/home/cspark/Work/simulation_codes-working/lume-astra/bin/astra
%env GENERATOR_BIN=/home/cspark/Work/simulation_codes-working/lume-astra/bin/generator
!echo $ASTRA_BIN
!echo $GENERATOR_BIN

env: ASTRA_BIN=/home/cspark/Work/simulation_codes-working/lume-astra/bin/astra
env: GENERATOR_BIN=/home/cspark/Work/simulation_codes-working/lume-astra/bin/generator
/home/cspark/Work/simulation_codes-working/lume-astra/bin/astra
/home/cspark/Work/simulation_codes-working/lume-astra/bin/generator


In [5]:
def astra_simulation(parameters, weights):
    # Create an Astra object
    astra_sim = Astra('astra.in')
    astra_sim.timeout = None
    astra_sim.verbose = False

    print ("parameters:", parameters)
    # Set up Astra parameters by modifying the input file or setting the attributes in the object
    astra_sim['solenoid:maxb(1)'] = parameters[0]           # solenoid strength [T]
    astra_sim['quadrupole:q_grad(1)'] = parameters[1]        # quadrupole strength [1/m^2]
    astra_sim['quadrupole:q_grad(2)'] = parameters[2]        # quadrupole strength [1/m^2]
    astra_sim['cavity:phi(1)'] = parameters[3]              # RF gun cavity input phase [deg]
    astra_sim['cavity:phi(2)'] = parameters[4]              # ACC1 cavity input phase [deg]
    astra_sim['cavity:phi(3)'] = parameters[4]              # ACC2 cavity input phase [deg]
    astra_sim['cavity:phi(4)'] = parameters[5]              # ACC3 cavity input phase [deg]
    astra_sim['cavity:phi(5)'] = parameters[5]              # ACC4 cavity input phase [deg]

    # Run the Astra simulation
    astra_sim.run()

    # Extract results from the simulation
    norm_emit_x = astra_sim.output['stats']['norm_emit_x'][-1]
    norm_emit_y = astra_sim.output['stats']['norm_emit_y'][-1]
    sigma_energy = astra_sim.output['stats']['sigma_energy'][-1]

    print ("objectives:", norm_emit_x, norm_emit_y, sigma_energy, "\n")

    target_norm_emit = 10e-9
    target_energy_spread = 0.005

    norm_emit0 = target_norm_emit * bg
    sigma_energy0 = 200.19e6 * target_energy_spread

    # weights
    w1, w2, w3 = weights
    
    # Objective: Minimize the sum of horizontal emittance, vertical emittance, and energy spread
    objectives = (w1 * norm_emit_x + w2 * norm_emit_y) / norm_emit0 + w3 * sigma_energy / sigma_energy0

    return objectives

In [6]:
def get_weights_from_filename(filename):
    """
    Extracts the list of weights from a filename generated by the optimization script.

    Args:
        filename (str): The full filename, e.g., 'data/optimization_results_0p05_0p05_0p9.csv'.

    Returns:
        list: A list of floats representing the weights.
    """
    # Define a pattern to find the weight part of the filename
    # This looks for any file that starts with "gp_minimize_result_" or "optimization_results_"
    # and ends with a file extension like .pkl or .csv
    pattern = r'(?:gp_minimize_result_|optimization_results_)(.*)\.(?:pkl|csv)'

    match = re.search(pattern, os.path.basename(filename))
    
    if match:
        weights_string = match.group(1)
        # Replace 'p' with '.' and split the string by underscores
        weight_parts = weights_string.replace('p', '.').split('_')
        
        # Convert each string part to a float and return as a list
        try:
            weights = [float(w) for w in weight_parts]
            return weights
        except ValueError:
            print(f"Error: Could not convert weight parts to floats: {weight_parts}")
            return None
    else:
        print(f"Error: Filename format not recognized: {filename}")
        return None

In [11]:
weight_combinations = [
    #[0.05, 0.05, 0.9],   # 0
    #[0.1, 0.1, 0.8],     # 1
    #[0.15, 0.15, 0.7],   # 2
    #[0.2, 0.2, 0.6],     # 3
    #[0.25, 0.25, 0.5],   # 4
    [0.3, 0.3, 0.4],     # 5
    [0.35, 0.35, 0.3],   # 6 
    [0.4, 0.4, 0.2],     # 7
    [0.45, 0.45, 0.1],   # 8
]

In [12]:
bg = 392.76205690637397 * 0.9999967587565166
print ("beta * gamma:", bg)

beta * gamma: 392.7607838689165


In [13]:
def run_astra_simulation(parameters):
    astra_run = Astra('astra.in')
    
    astra_run['solenoid:maxb(1)'] = parameters[0]           # solenoid strength [T]
    astra_run['quadrupole:q_grad(1)'] = parameters[1]        # quadrupole strength [1/m^2]
    astra_run['quadrupole:q_grad(2)'] = parameters[2]        # quadrupole strength [1/m^2]
    astra_run['cavity:phi(1)'] = parameters[3]              # RF gun cavity input phase [deg]
    astra_run['cavity:phi(2)'] = parameters[4]              # ACC1 cavity input phase [deg]
    astra_run['cavity:phi(3)'] = parameters[4]              # ACC2 cavity input phase [deg]
    astra_run['cavity:phi(4)'] = parameters[5]              # ACC3 cavity input phase [deg]
    astra_run['cavity:phi(5)'] = parameters[5]              # ACC4 cavity input phase [deg]

    astra_run.timeout = None
    astra_run.verbose = False
    astra_run.run()

    norm_emit_x = astra_run.output['stats']['norm_emit_x'][-1]
    norm_emit_y = astra_run.output['stats']['norm_emit_y'][-1]
    sigma_energy = astra_run.output['stats']['sigma_energy'][-1]

    target_norm_emit = 10e-9
    target_energy_spread = 0.005

    norm_emit0 = target_norm_emit * bg
    sigma_energy0 = 200.19e6 * target_energy_spread
    
    w1, w2, w3 = weights
    objectives = (w1 * norm_emit_x + w2 * norm_emit_y) / norm_emit0 + w3 * sigma_energy / sigma_energy0

    print ("objectives:", norm_emit_x, norm_emit_y, sigma_energy, objectives)

    return (astra_run.output['stats'])

In [ ]:
num_processes = 8 #mp.cpu_count()
print(f"Running simulations on {num_processes} cores...")

for weights in weight_combinations:
    print(f"Running optimization for weights: {weights}\n")
    filename_weights = "_".join(map(lambda x: str(x).replace('.', 'p'), weights))
    pickle_filename = f'data/gp_minimize_result_{filename_weights}.pkl'
    csv_filename = f'data/optimization_results_{filename_weights}.csv'
    
    with open(pickle_filename, 'rb') as f:
        result = pickle.load(f)
    
    df_results = pd.read_csv(csv_filename)
    
    with mp.Pool(num_processes) as pool:
        # Convert the DataFrame rows to a list of lists for the pool.map function
        parameter_list_of_lists = df_results.values.tolist()
        
        # Use pool.map to apply the run_simulation function to each list of parameters
        # The results will be gathered in the order of the input list
        all_output_list = pool.map(run_astra_simulation, parameter_list_of_lists)

    print(f"Simultions Done")
    
    output_filename = f'results/output_{filename_weights}.pkl'
    with open(output_filename, 'wb') as f:
        pickle.dump(all_output_list, f)

Running simulations on 8 cores...
Running optimization for weights: [0.3, 0.3, 0.4]

objectives: 0.00019418999999999998 7.587699999999999e-05 10444000.0 24.801992899011054
objectives: 2.5454999999999996e-05 2.5414999999999996e-05 3588400.0 5.319568926904114
objectives: 3.6247999999999996e-05 4.6034999999999995e-05 3938100.0 7.8587155946290395
objectives: 9.9964e-05 2.8431999999999998e-05 4043100.0 11.422896017254661
objectives: 6.1863e-05 0.00022936 10870000.0 26.588176618497403
objectives: 8.1906e-05 6.4011e-05 373920.0 11.294912508745142
objectives: 9.1363e-05 6.7691e-05 8183800.0 15.41933478194932
objectives: 9.8168e-05 0.00037697 6192800.0 38.76693607330396
objectives: 6.792199999999999e-05 7.1351e-05 3866600.0 12.18317407552103
objectives: 7.5808e-05 6.145099999999999e-05 1979100.0 11.275056549113074
objectives: 3.9172e-05 7.1043e-05 2461900.0 9.40230841156085
objectives: 0.00017771 5.3268999999999994e-05 3579500.0 19.07316480644733
objectives: 5.545e-05 5.5684e-05 904680.0 8.8502